## This Notebook will be used to create operational tables for analytical purposes

In [1]:
#importing necessary libraries and setting up paths. do not update this cell. Run first before running any other cell. 
import duckdb
from pathlib import Path

processed_path = Path("../data/processed")
analytics_path = Path("../data/analytics")

visitors_path = (processed_path / "visitors.parquet").as_posix()
sessions_path = (processed_path / "sessions.parquet").as_posix()
outcomes_path = (processed_path / "questionnaire_outcomes.parquet").as_posix()
events_path = (processed_path / "questionnaire_events.parquet").as_posix()
answers_path = (processed_path / "questionnaire_answers.parquet").as_posix()
questions_path = (processed_path / "questionnaire_questions.parquet").as_posix()

In [2]:
#Create visitor analytics dataframe

visitor_analytics = duckdb.sql(f"""
SELECT
    v.visitor_id,
    2026 - v.birth_year AS age,
    CASE
        WHEN v.birth_year IS NULL THEN 'not provided'
        WHEN 2026 - v.birth_year < 25 THEN '16-24'
        WHEN 2026 - v.birth_year < 35 THEN '25-34'
        WHEN 2026 - v.birth_year < 45 THEN '35-44'
        WHEN 2026 - v.birth_year < 55 THEN '45-54'
        WHEN 2026 - v.birth_year < 65 THEN '55-64'
        ELSE '65+'
    END AS age_group,
    v.gender,
    v.region,
    COUNT(DISTINCT s.session_id) AS total_sessions,
    COUNT(DISTINCT o.session_id) AS completed_sessions,
    SUM(CASE WHEN s.is_returning_visitor THEN 1 ELSE 0 END) AS returning_sessions,
    MIN(s.session_started_at) AS first_session,
    MAX(s.session_started_at) AS last_session
FROM read_parquet('{visitors_path}') v
LEFT JOIN read_parquet('{sessions_path}') s USING(visitor_id)
LEFT JOIN read_parquet('{outcomes_path}') o USING(session_id)
GROUP BY v.visitor_id, v.birth_year, v.gender, v.region
""").df()

In [3]:
#create session analytics dataframe

session_analytics = duckdb.sql(f"""
WITH event_summary AS (
    SELECT
        session_id,
        MIN(CASE WHEN event_type = 'questionnaire_start' THEN event_timestamp END) AS started_at,
        MIN(CASE WHEN event_type = 'questionnaire_complete' THEN event_timestamp END) AS completed_at
    FROM read_parquet('{events_path}')
    GROUP BY session_id
),
answer_summary AS (
    SELECT session_id, COUNT(DISTINCT question_number) AS answered_questions
    FROM read_parquet('{answers_path}')
    GROUP BY session_id
)

SELECT
    s.session_id,
    s.visitor_id,
    2026 - v.birth_year AS age,
    v.gender,
    v.region,
    s.acquisition_source,
    s.campaign_name,
    s.device_type,
    s.landing_page,
    s.is_returning_visitor,
    s.session_started_at,
    e.started_at IS NOT NULL AS started,
    e.completed_at IS NOT NULL AS completed,
    COALESCE(a.answered_questions, 0) AS answered_questions,
    o.outcome_category,
    DATE_DIFF('second', e.started_at, e.completed_at) / 60.0 AS completion_minutes
FROM read_parquet('{sessions_path}') s
LEFT JOIN read_parquet('{visitors_path}') v USING(visitor_id)
LEFT JOIN event_summary e USING(session_id)
LEFT JOIN answer_summary a USING(session_id)
LEFT JOIN read_parquet('{outcomes_path}') o USING(session_id)
""").df()

In [4]:
#create question analytics dataframe

question_analytics = duckdb.sql(f"""
WITH question_views AS (
    SELECT
        session_id,
        visitor_id,
        question_number,
        MIN(event_timestamp) AS first_viewed_at,
        MAX(event_timestamp) AS last_viewed_at
    FROM read_parquet('{events_path}')
    WHERE event_type = 'question_view'
    GROUP BY session_id, visitor_id, question_number
),

final_answers AS (
    SELECT
        session_id,
        visitor_id,
        question_id,
        question_number,
        answer_value,
        answered_at
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY session_id, question_number
                ORDER BY answered_at DESC
            ) AS rn
        FROM read_parquet('{answers_path}')
    )
    WHERE rn = 1
)

SELECT
    qv.session_id,
    qv.visitor_id,
    qv.question_number,
    q.question_id,
    q.question_text,

    qv.first_viewed_at,
    qv.last_viewed_at,

    fa.answer_value,
    fa.answered_at,

    fa.answer_value IS NOT NULL AS answered,

    DATE_DIFF(
        'second',
        qv.first_viewed_at,
        fa.answered_at
    ) AS answer_time_seconds,

    s.campaign_name,
    s.acquisition_source,
    s.device_type,

    o.outcome_category

FROM question_views qv

LEFT JOIN read_parquet('{questions_path}') q
    ON qv.question_number = q.question_number

LEFT JOIN final_answers fa
    ON qv.session_id = fa.session_id
    AND qv.question_number = fa.question_number

LEFT JOIN read_parquet('{sessions_path}') s
    ON qv.session_id = s.session_id

LEFT JOIN read_parquet('{outcomes_path}') o
    ON qv.session_id = o.session_id

ORDER BY qv.session_id, qv.question_number
""").df()

In [5]:
#transform and save analytics dataframes to parquet files

visitor_analytics.to_parquet(
    analytics_path / "visitor_analytics.parquet",
    index=False
)

session_analytics.to_parquet(
    analytics_path / "session_analytics.parquet",
    index=False
)
question_analytics.to_parquet(
    analytics_path / "question_analytics.parquet",
    index=False
)

In [6]:
display(session_analytics.head())
print(session_analytics.shape)

,session_id,visitor_id,age,gender,region,acquisition_source,campaign_name,device_type,landing_page,is_returning_visitor,session_started_at,started,completed,answered_questions,outcome_category,completion_minutes
0,SES_00000060,VIS_00010111,29,male,occitanie,chatgpt,NaN,mobile,/health/diabetes-awareness,False,2026-06-01 06:19:57.185978+02:00,True,True,5,no_current_indication,1.316667
1,SES_00000135,VIS_00000155,25,female,occitanie,chatgpt,NaN,mobile,/health/diabetes-awareness,False,2026-06-01 11:11:15.102774+02:00,True,True,5,no_current_indication,2.116667
2,SES_00000178,VIS_00010868,31,female,bretagne,google-organic,NaN,mobile,/health/diabetes-awareness,False,2026-06-01 14:16:10.269500+02:00,True,True,5,possible_risk,1.316667
3,SES_00000187,VIS_00004243,49,male,grand-est,google-organic,NaN,desktop,/health/diabetes-awareness,False,2026-06-01 14:45:02.752330+02:00,True,True,5,no_current_indication,2.133333
4,SES_00000192,VIS_00018738,18,female,hauts-de-france,social,diabetes_social_awareness,desktop,/health/diabetes-awareness,False,2026-06-01 15:07:52.952772+02:00,True,True,5,no_current_indication,1.383333


(25475, 16)


In [7]:
display(visitor_analytics.head())
print(visitor_analytics.shape)

,visitor_id,age,age_group,gender,region,total_sessions,completed_sessions,returning_sessions,first_session,last_session
0,VIS_00016978,32,25-34,female,auvergne-rhone-alpes,1,1,0.0,2026-06-02 02:06:19.474651+02:00,2026-06-02 02:06:19.474651+02:00
1,VIS_00012737,50,45-54,male,grand-est,1,1,0.0,2026-06-02 04:54:26.259809+02:00,2026-06-02 04:54:26.259809+02:00
2,VIS_00004644,41,35-44,male,grand-est,1,1,0.0,2026-06-02 06:39:57.464353+02:00,2026-06-02 06:39:57.464353+02:00
3,VIS_00003689,45,45-54,male,hauts-de-france,1,1,0.0,2026-06-02 06:45:28.104909+02:00,2026-06-02 06:45:28.104909+02:00
4,VIS_00011738,45,45-54,male,provence-alpes-cote-d'azur,3,1,2.0,2026-06-02 08:10:18.024862+02:00,2026-07-13 17:25:09.273664+02:00


(19825, 10)


In [8]:
display(question_analytics.head())
print(question_analytics.shape)

,session_id,visitor_id,question_number,question_id,question_text,first_viewed_at,last_viewed_at,answer_value,answered_at,answered,answer_time_seconds,campaign_name,acquisition_source,device_type,outcome_category
0,SES_00000001,VIS_00016128,1,diabetes_q1,Have you ever been told by a healthcare profes...,2026-06-01 02:02:11.208955+02:00,2026-06-01 02:02:11.208955+02:00,no,2026-06-01 02:02:35.588484+02:00,True,24,diabetes_social_awareness,social,desktop,NaN
1,SES_00000001,VIS_00016128,2,diabetes_q2,"During the last 3 months, have you frequently ...",2026-06-01 02:02:36.940089+02:00,2026-06-01 02:02:36.940089+02:00,NaN,NaT,False,<NA>,diabetes_social_awareness,social,desktop,NaN
2,SES_00000002,VIS_00012147,1,diabetes_q1,Have you ever been told by a healthcare profes...,2026-06-01 02:08:58.447400+02:00,2026-06-01 02:08:58.447400+02:00,no,2026-06-01 02:09:23.966499+02:00,True,25,NaN,google-organic,desktop,NaN
3,SES_00000002,VIS_00012147,2,diabetes_q2,"During the last 3 months, have you frequently ...",2026-06-01 02:09:27.652554+02:00,2026-06-01 02:09:27.652554+02:00,no,2026-06-01 02:09:48.773896+02:00,True,21,NaN,google-organic,desktop,NaN
4,SES_00000002,VIS_00012147,3,diabetes_q3,Has a parent or sibling been diagnosed with Ty...,2026-06-01 02:09:49.972772+02:00,2026-06-01 02:09:49.972772+02:00,NaN,NaT,False,<NA>,NaN,google-organic,desktop,NaN


(69973, 15)
